In [1]:
#Uses the OpenTopography API (https://portal.opentopography.org/apidocs/#/Public/getGlobalDem)
# requires API key in yaml file

# slope and aspect calculated with https://richdem.readthedocs.io/en/latest/python_api.html#richdem.TerrainAttribute
# methodology: https://citeseerx.ist.psu.edu/document?repid=rep1&type=pdf&doi=4332a821c684169db4af6bc35836ad12973b3148

In [6]:
import os
import geopandas as gpd
import pandas as pd
import fiona
import pyproj
from osgeo import gdal
from PIL import Image
import georasters as gr
import matplotlib.pyplot as plt

import requests
from requests.auth import HTTPBasicAuth
import yaml

import elevation
import richdem as rd

import rasterio
from rasterio.plot import show
from rasterstats import zonal_stats

In [7]:
config_path = "../data/opentopo_api.yaml"
with open(config_path) as conf_file:
    config = yaml.safe_load(conf_file)
data_path = "../data/all_projects_TM_20240614/all_projects_TM.shp"

In [8]:
shapefile_features =[]
with fiona.collection(data_path, 'r') as shp_in:
    shp_crs = pyproj.crs.CRS(shp_in.crs)
    shpschema = shp_in.schema.copy()
    for obj in shp_in:
        shapefile_features.append(obj)
shp_gdf = gpd.GeoDataFrame.from_features([feature for feature in shapefile_features], crs=shp_crs)

In [9]:
project_list = shp_gdf['Project'].unique()

In [10]:
all_projects = {}
for project in project_list:
    project_gdf = shp_gdf[shp_gdf['Project'] == project]
    project_bounds = project_gdf.total_bounds
    query = {'demtype': 'NASADEM',
          'south': str(project_bounds[1]),
          'north': str(project_bounds[3]),
          'west': str(project_bounds[0]),
          'east': str(project_bounds[2]),
          'outputFormat': 'GTiff',
          'API_Key': str(config['key'])}
    with open('../data/temp.tiff', 'wb') as f:
        ret = requests.get(config['dem_url'], stream=True, params=query)
        for data in ret.iter_content(1024):
            f.write(data)
    print(f"DEM downloaded for {project}")
    dem = rd.LoadGDAL('../data/temp.tiff')
    slope = rd.TerrainAttribute(dem, attrib='slope_riserun')
    aspect = rd.TerrainAttribute(dem, attrib='aspect')
    dem_r = rasterio.open('temp.tiff')
    affine = dem_r.transform

    project_dict = project_gdf.to_dict('records')

    for i in range(0, len(project_dict)):
        poly = project_dict[i]['geometry']
        project_dict[i]['slope_stats']= zonal_stats(poly, slope, affine=affine, stats=["min", "max", "mean", "median", "majority"])
        project_dict[i]['aspect_stats']= zonal_stats(poly, aspect, affine=affine, stats=["min", "max", "mean", "median", "majority"])
    all_projects[project] = project_dict

DEM downloaded for addax-gum



A Slope calculation (rise/run)
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.000158735==================   ] (94% - 0.0s - 1 threads)

A Aspect attribute calculation
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.000650569==================   ] (94% - 0.0s - 1 threads)
/home/rlrognstad/miniconda3/envs/geopandas/lib/python3.9/site-packages/rasterstats/io.py:328: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


DEM downloaded for adhd



A Slope calculation (rise/run)
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.00336662===================== ] (99% - 0.0s - 1 threads)

A Aspect attribute calculation
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.0124056====================== ] (99% - 0.0s - 1 threads)


DEM downloaded for aerobic-agroforestry


...


A Slope calculation (rise/run)
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.000144822==================== ] (98% - 0.0s - 1 threads)

A Aspect attribute calculation
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.000195536==================== ] (98% - 0.0s - 1 threads)


DEM downloaded for wildlife-action-group



A Slope calculation (rise/run)
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.00152421===================== ] (99% - 0.0s - 1 threads)

A Aspect attribute calculation
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.00356236===================== ] (99% - 0.0s - 1 threads)


DEM downloaded for world-vision-intl-ethiopia



A Slope calculation (rise/run)
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.00360736===================== ] (99% - 0.0s - 1 threads)

A Aspect attribute calculation
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.0145162====================== ] (99% - 0.0s - 1 threads)


In [11]:
df_list = []
for key in all_projects.keys():
    project_df = pd.DataFrame.from_dict(all_projects[key], orient='columns')
    df_list.append(project_df)

all_projects_df = pd.concat(df_list)

In [13]:
all_projects_df.columns

Index(['geometry', 'Project', 'SiteName', 'poly_name', 'plantstart',
       'plantend', 'distr', 'num_trees', 'target_sys', 'practice', 'bounds',
       'slope_stats', 'aspect_stats'],
      dtype='object')

In [14]:
all_projects_df = all_projects_df[['geometry', 'Project', 'SiteName', 'poly_name', 'slope_stats', 'aspect_stats']]

In [16]:
(all_projects_df.loc[:, all_projects_df.columns != 'geometry']).to_csv("../data/bucketing_output/polygon_slope_aspect.csv")